# PostgreSQL Data Loading

## Objective

This notebook connects the cleaned e-commerce dataset to PostgreSQL and loads the processed data into a relational database for SQL analysis and Power BI reporting.

In [1]:
import pandas as pd
from pandas import errors
from sqlalchemy import create_engine
from sqlalchemy.engine import URL


In [2]:
file_path = "../data/processed/ecommerce_cleaned.csv"

In [3]:
df = pd.read_csv(file_path)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 1,000,123
Columns: 68


In [4]:

date_columns = ['order_date', 'account_creation_date']

for column in date_columns:
    df[column] = pd.to_datetime(df[column],
    errors='coerce')

df[date_columns].head(3)

,order_date,account_creation_date
0,2024-10-14 11:20:05.496679,2022-04-23
1,2024-10-21 00:49:44.681065,2022-08-22
2,2025-03-17 19:49:36.983317,2024-08-04


```Creating a PostgreSQL connection```

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=int(os.getenv("PG_PORT")),
    database=os.getenv("PG_DATABASE")
)

engine = create_engine(connection_url)

```Testing the connection```

In [6]:
from sqlalchemy import text

try:
    with engine.connect() as connection:
        result = connection.execute(
            text("SELECT version();")
        )

        print(result.fetchone())

except Exception as error:
    print("Database connection failed:")
    print(error)

('PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35223, 64-bit',)


```Loading the DataFrame into PostgreSQL```


In [7]:
table_name = "ecommerce_orders"

df.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi"
)

print("Data successfully loaded into PostgreSQL.")

Data successfully loaded into PostgreSQL.


```Validateating the imported data```


In [8]:
query = """
SELECT
    COUNT(*) AS total_rows
FROM ecommerce_orders;
"""

pd.read_sql(query, engine)

,total_rows
0,1000123


In [9]:
query = """
SELECT *
FROM ecommerce_orders
LIMIT 5;
"""

pd.read_sql(query, engine)

,order_id,order_date,order_year,order_month,order_day,order_hour,order_minute,order_second,is_weekend,order_status,...,fraud_risk_score,profit_margin_percent,order_priority,support_ticket_created,profit_per_unit,revenue_per_unit,age_group,month_name,quarter,high_value_order
0,ORD-XAJI0,2024-10-14 11:20:05.496679,2024,10,14,11,20,5,False,Completed,...,45.3,28.66,Medium,Yes,31.08,108.46,46-55,October,4,False
1,ORD-NHJ7X,2024-10-21 00:49:44.681065,2024,10,21,0,49,44,False,Completed,...,97.1,33.91,Low,Yes,17.39,51.28,56-65,October,4,False
2,ORD-YTJXE,2025-03-17 19:49:36.983317,2025,3,17,19,49,36,False,Completed,...,43.8,52.77,High,No,156.15,295.89,46-55,March,1,False
3,ORD-EIMVI,2024-09-27 06:24:44.913768,2024,9,27,6,24,44,False,Completed,...,96.9,38.15,Low,Yes,71.09,186.33,46-55,September,3,False
4,ORD-OR56F,2025-05-21 17:10:48.401882,2025,5,21,17,10,48,False,Completed,...,45.8,33.05,High,Yes,127.05,384.44,26-35,May,2,False
